In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from IPython.display import display

print("numpy     ", np.__version__)
print("pandas    ", pd.__version__)
print("matplotlib", mpl.__version__)
print("seaborn   ", sns.__version__)

mpl.rcParams["figure.dpi"] = 100
mpl.rcParams["figure.figsize"] = (7, 4)
rng = np.random.default_rng(42)
os.makedirs("figures", exist_ok=True)
os.makedirs("data", exist_ok=True)

In [ ]:
anscombe = sns.load_dataset("anscombe")

stats = anscombe.groupby("dataset")[["x", "y"]].agg(["mean", "std"]).round(2)
stats["corr(x,y)"] = anscombe.groupby("dataset")[["x", "y"]].apply(lambda g: g["x"].corr(g["y"])).round(3)
stats

In [ ]:
sns.lmplot(data=anscombe, x="x", y="y", col="dataset", col_wrap=2,
           height=3, ci=None, scatter_kws={"s": 50})
plt.show()

### `matplotlib`

In [ ]:
x = np.linspace(0, 2 * np.pi, 200)

# Стиль 1: pyplot (неявный)
plt.plot(x, np.sin(x))
plt.title("pyplot-стиль")
plt.show()

In [ ]:
# Стиль 2: объектно-ориентированный — используем его дальше
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot(x, np.sin(x), label="sin(x)")
ax.plot(x, np.cos(x), label="cos(x)", linestyle="--")
ax.set(title="OO-стиль", xlabel="x, рад", ylabel="y")
ax.legend()
ax.grid(alpha=0.3)
plt.show()

# Заглянем внутрь объектов
print("fig:", type(fig).__name__, "| axes в фигуре:", fig.axes)
print("линий на ax:", len(ax.lines), "| пределы по x:", ax.get_xlim())

### `plot`, `scatter`, `bar`, `hist`

In [ ]:
fig, axs = plt.subplots(2, 2, figsize=(10, 7), constrained_layout=True)

# 1) линия — временной ряд (случайное блуждание)
t = np.arange(60)
axs[0, 0].plot(t, np.cumsum(rng.normal(0.3, 1, 60)))
axs[0, 0].set_title("plot: временной ряд")

# 2) scatter — цвет и размер несут дополнительные переменные
a = rng.normal(0, 1, 200)
b = 0.6 * a + rng.normal(0, 0.8, 200)
c = rng.uniform(10, 100, 200)
sc = axs[0, 1].scatter(a, b, c=c, s=c, alpha=0.6, cmap="viridis")
fig.colorbar(sc, ax=axs[0, 1], label="третья переменная")
axs[0, 1].set_title("scatter: x, y, цвет, размер")

# 3) bar — сравнение категорий
cats, vals = ["A", "B", "C", "D"], [23, 17, 35, 29]
axs[1, 0].bar(cats, vals, color="tab:blue")
axs[1, 0].set_title("bar: категории")

# 4) hist — распределение
axs[1, 1].hist(rng.normal(170, 10, 1000), bins=30, edgecolor="white")
axs[1, 1].set_title("hist: рост, см")

plt.show()

### color, annotations

In [ ]:
dates = pd.date_range("2024-01-01", periods=365, freq="D")
k = np.arange(365)
sales = pd.Series(100 + 0.1 * k + 15 * np.sin(2 * np.pi * k / 7) + rng.normal(0, 5, 365), index=dates)
rolling = sales.rolling(7).mean()

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(sales.index, sales, color="lightgray", label="ежедневно")
ax.plot(rolling.index, rolling, color="tab:red", lw=2, label="скользящее среднее, 7 дней")

peak = rolling.idxmax()
ax.annotate(f"максимум: {rolling.max():.0f}",
            xy=(peak, rolling.max()),
            xytext=(peak - pd.Timedelta(days=100), rolling.max() + 12),
            arrowprops=dict(arrowstyle="->"))
ax.axvspan(pd.Timestamp("2024-06-01"), pd.Timestamp("2024-08-31"), color="orange", alpha=0.12, label="лето")

ax.xaxis.set_major_locator(mdates.MonthLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b"))
ax.set(title="Продажи за 2024 год", ylabel="шт. в день")
ax.legend(loc="upper left")
plt.show()

In [ ]:
# Логарифмическая шкала и вторая ось Y
fig, axs = plt.subplots(1, 2, figsize=(10, 3.5), constrained_layout=True)

years = np.arange(2000, 2025)
users = 1000 * 1.35 ** (years - 2000)          # экспоненциальный рост
axs[0].plot(years, users)
axs[0].set_yscale("log")                        # на логшкале экспонента — прямая
axs[0].set(title="Логарифмическая шкала Y", ylabel="пользователи")

temp = 10 + 12 * np.sin(2 * np.pi * (np.arange(12) - 3) / 12)
rain = 40 + 20 * np.cos(2 * np.pi * np.arange(12) / 12)
axs[1].bar(range(1, 13), rain, color="lightsteelblue")
axs[1].set(title="Две оси: осторожно!", xlabel="месяц", ylabel="осадки, мм")
ax2 = axs[1].twinx()
ax2.plot(range(1, 13), temp, color="tab:red", marker="o")
ax2.set_ylabel("температура, °C", color="tab:red")
plt.show()

### Раскладки: subplots, subplot_mosaic

In [ ]:
fig, axd = plt.subplot_mosaic(
    [["hx", "."],
     ["main", "hy"]],
    width_ratios=[4, 1], height_ratios=[1, 4],
    figsize=(6, 6), constrained_layout=True)

u = rng.normal(0, 1, 500)
v = 0.5 * u + rng.normal(0, 0.8, 500)

axd["main"].scatter(u, v, s=12, alpha=0.6)
axd["hx"].hist(u, bins=30, color="gray")
axd["hy"].hist(v, bins=30, orientation="horizontal", color="gray")
axd["hx"].sharex(axd["main"]); axd["hy"].sharey(axd["main"])
axd["hx"].tick_params(labelbottom=False); axd["hy"].tick_params(labelleft=False)
axd["main"].set(xlabel="u", ylabel="v")
plt.show()

### Стили

In [ ]:
print(plt.style.available[:8], "...")

for style in ["default", "ggplot", "fivethirtyeight"]:
    with plt.style.context(style):
        fig, ax = plt.subplots(figsize=(5, 2.4))
        ax.plot(x, np.sin(x)); ax.plot(x, np.cos(x))
        ax.set_title(f"style = {style}")
        plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3))
ax.plot(x, np.sin(x))
ax.set_title("Сохраняем в разные форматы")

fig.savefig("figures/demo.png", dpi=200, bbox_inches="tight")   # растр
fig.savefig("figures/demo.svg", bbox_inches="tight")            # вектор
fig.savefig("figures/demo.pdf", bbox_inches="tight")            # вектор для LaTeX/статьи
plt.close(fig)
print(sorted(os.listdir("figures")))

### Пример

In [ ]:
months = ["Янв", "Фев", "Мар", "Апр", "Май", "Июн"]
revenue = pd.DataFrame({
    "Москва": [120, 135, 150, 148, 170, 190],
    "Казань": [60, 64, 71, 80, 79, 92],
    "Томск":  [30, 28, 35, 40, 38, 45],
}, index=months)
revenue

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4), constrained_layout=True)

for city in revenue.columns:
    ax1.plot(revenue.index, revenue[city], marker="o", label=city)
ax1.set(title="Динамика по месяцам", ylabel="выручка, млн ₽")
ax1.legend()

total = revenue.sum().sort_values()
bars = ax2.barh(total.index, total.values, color="tab:blue")
ax2.bar_label(bars, padding=3)
ax2.set(title="Итого за полгода", xlabel="выручка, млн ₽")
ax2.set_xlim(0, total.max() * 1.15)

fig.suptitle("Выручка по городам, I полугодие", fontsize=14)
fig.savefig("figures/task1.png", dpi=150, bbox_inches="tight")
plt.show()

### `seaborn`

In [ ]:
sns.set_theme(style="whitegrid", context="notebook", palette="deep")   # единая тема для всех графиков

wide = pd.DataFrame({"month": ["Янв", "Фев", "Мар"], "Москва": [120, 135, 150], "Казань": [60, 64, 71]})
long = wide.melt(id_vars="month", var_name="city", value_name="revenue")   # wide -> long
display(wide); display(long)

In [ ]:
penguins = sns.load_dataset("penguins")
tips = sns.load_dataset("tips")
flights = sns.load_dataset("flights")

display(penguins.head(3)); display(tips.head(3))
print("Пропуски в penguins:\n", penguins.isna().sum()[lambda s: s > 0])

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(10, 3.5))
sns.histplot(data=penguins, x="body_mass_g", ax=axs[0])
sns.boxplot(data=penguins, x="species", y="body_mass_g", ax=axs[1])
plt.show()

# figure-level: сам создаёт фигуру и сетку по col=
g = sns.displot(data=penguins, x="body_mass_g", col="species", height=3, aspect=1)
print(type(g))

### Распределения: `histplot`, `kdeplot`, `ecdfplot`

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(12, 3), sharey=True)
for ax, bins in zip(axs, [5, 20, 60]):
    sns.histplot(data=penguins, x="flipper_length_mm", bins=bins, ax=ax)
    ax.set_title(f"bins = {bins}")
plt.show()

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(14, 3.8))

sns.histplot(data=penguins, x="body_mass_g", hue="species", element="step",
             stat="density", common_norm=False, ax=axs[0])          # нормировка внутри групп
axs[0].set_title("histplot + hue")

sns.kdeplot(data=penguins, x="body_mass_g", hue="species", fill=True, alpha=0.4, ax=axs[1])
axs[1].set_title("kdeplot (сглаженная плотность)")

sns.ecdfplot(data=penguins, x="body_mass_g", hue="species", ax=axs[2])
axs[2].set_title("ecdfplot (эмпирическая CDF)")
plt.tight_layout()
plt.show()

In [ ]:
# Двумерная плотность
sns.kdeplot(data=penguins, x="bill_length_mm", y="bill_depth_mm", hue="species", levels=5)
plt.title("2D KDE: размеры клюва")
plt.show()

### Категориальные данные: `boxplot`, `violinplot`, `stripplot`, `barplot`, `countplot`

In [ ]:
fig, axs = plt.subplots(2, 3, figsize=(15, 8), constrained_layout=True)

sns.countplot(data=tips, x="day", hue="time", ax=axs[0, 0]);                       axs[0, 0].set_title("countplot")
sns.barplot(data=tips, x="day", y="total_bill", ax=axs[0, 1]);                      axs[0, 1].set_title("barplot: среднее ± 95% ДИ")
sns.boxplot(data=tips, x="day", y="total_bill", hue="sex", ax=axs[0, 2]);           axs[0, 2].set_title("boxplot")
sns.violinplot(data=tips, x="day", y="total_bill", hue="sex", split=True,
               inner="quart", ax=axs[1, 0]);                                         axs[1, 0].set_title("violinplot (split)")
sns.stripplot(data=tips, x="day", y="total_bill", hue="sex", dodge=True,
              alpha=0.6, ax=axs[1, 1]);                                              axs[1, 1].set_title("stripplot")
sns.boxplot(data=tips, x="day", y="total_bill", color="white", showfliers=False, ax=axs[1, 2])
sns.stripplot(data=tips, x="day", y="total_bill", color="black", alpha=0.4, size=3, ax=axs[1, 2])
axs[1, 2].set_title("boxplot + stripplot: форма и точки")
plt.show()

In [ ]:
# figure-level версия: catplot с сеткой по col
g = sns.catplot(data=tips, x="day", y="tip", hue="smoker", col="time",
                kind="box", height=3.8, aspect=1.1, sharex=False)   # sharex=False: у каждой панели свои дни
g.set_axis_labels("День недели", "Чаевые, $")
g.set_titles("Время: {col_name}")
plt.show()

### Связи между переменными: `scatterplot`, `lineplot`, `regplot`/`lmplot`, `relplot`

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(13, 4.5))

sns.scatterplot(data=tips, x="total_bill", y="tip", hue="time", size="size",
                sizes=(20, 200), alpha=0.7, ax=axs[0])
axs[0].set_title("scatterplot: hue + size")

sns.lineplot(data=flights, x="year", y="passengers", ax=axs[1])
axs[1].set_title("lineplot: среднее по 12 месяцам + ДИ")
plt.tight_layout(); plt.show()

In [ ]:
sns.lmplot(data=tips, x="total_bill", y="tip", hue="smoker", col="time", height=3.8)
plt.show()

sns.relplot(data=flights, x="year", y="passengers", hue="month", kind="line",
            palette="viridis", height=4, aspect=1.6)
plt.show()

### Матрицы: `heatmap`, корреляции

In [ ]:
corr = penguins.corr(numeric_only=True)
mask = np.triu(np.ones_like(corr, dtype=bool))          

fig, axs = plt.subplots(1, 2, figsize=(14, 4.5), gridspec_kw={"width_ratios": [1, 1.3]})
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="vlag", center=0,
            vmin=-1, vmax=1, square=True, ax=axs[0])
axs[0].set_title("Корреляции (расходящаяся палитра с центром в 0)")

pivot = flights.pivot(index="month", columns="year", values="passengers")
sns.heatmap(pivot, cmap="YlGnBu", ax=axs[1], cbar_kws={"label": "пассажиров"})
axs[1].set_title("flights: месяц × год (последовательная палитра)")
plt.tight_layout(); plt.show()

### `pairplot`, `jointplot`

In [ ]:
sns.pairplot(penguins, hue="species", corner=True, height=2.1,
             vars=["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"])
plt.show()

In [ ]:
sns.jointplot(data=penguins, x="flipper_length_mm", y="body_mass_g", hue="species", height=5)
plt.show()

In [ ]:
# Парадокс Симпсона: длина vs глубина клюва
fig, axs = plt.subplots(1, 2, figsize=(12, 4.5), sharey=True)

sns.regplot(data=penguins, x="bill_length_mm", y="bill_depth_mm", ax=axs[0], scatter_kws={"alpha": 0.5})
axs[0].set_title("Все пингвины вместе: связь ОТРИЦАТЕЛЬНА")

for sp, part in penguins.groupby("species"):
    sns.regplot(data=part, x="bill_length_mm", y="bill_depth_mm", ax=axs[1],
                label=sp, scatter_kws={"alpha": 0.5})
axs[1].set_title("Внутри каждого вида: связь ПОЛОЖИТЕЛЬНА")
axs[1].legend()
plt.tight_layout(); plt.show()

### Темы и палитры

In [ ]:
palettes = {
    "качественная: deep":         sns.color_palette("deep", 8),
    "качественная: colorblind":   sns.color_palette("colorblind", 8),
    "последовательная: rocket":   sns.color_palette("rocket", 8),
    "последовательная: viridis":  sns.color_palette("viridis", 8),
    "расходящаяся: vlag":         sns.color_palette("vlag", 8),
}
for name, pal in palettes.items():
    print(name)
    display(pal)

In [ ]:
fig = plt.figure(figsize=(10, 6))
for i, style in enumerate(["white", "whitegrid", "darkgrid", "ticks"], start=1):
    with sns.axes_style(style):                    # временная тема только для этого Axes
        ax = fig.add_subplot(2, 2, i)
        sns.lineplot(x=x, y=np.sin(x), ax=ax)
        ax.set_title(f'style="{style}"')
        if style == "ticks":
            sns.despine(ax=ax)                     # убрать верхнюю и правую рамки
plt.tight_layout(); plt.show()

### `seaborn` + `matplotlib`

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
sns.barplot(data=tips, x="day", y="total_bill", errorbar=None, color="steelblue", ax=ax)
for container in ax.containers:
    ax.bar_label(container, fmt="%.1f", padding=3)       # метод matplotlib на результате seaborn
ax.set(title="Средний счёт по дням", xlabel="", ylabel="$")
sns.despine()
plt.show()

In [ ]:
import seaborn.objects as so

(
    so.Plot(penguins, x="flipper_length_mm", y="body_mass_g", color="species")
    .add(so.Dot(alpha=0.6))
    .add(so.Line(), so.PolyFit(order=1))
    .label(x="Длина ласт, мм", y="Масса, г", color="Вид")
)

In [ ]:
# (1) Обрезанная ось столбцов — визуально втрое «лучше» при разнице в 2%
vals = {"Продукт A": 96, "Продукт B": 97, "Продукт C": 98}
fig, axs = plt.subplots(1, 2, figsize=(10, 3.6), constrained_layout=True)
for ax, ymin, title in zip(axs, [95, 0], ["Обрезанная ось: «C втрое лучше A!»", "Ось от нуля: разница ~2%"]):
    ax.bar(list(vals), list(vals.values()), color=["#999999", "#999999", "tab:blue"])
    ax.set_ylim(ymin, 100)
    ax.set_title(title, fontsize=11)
plt.show()

In [ ]:
# (2) Круговая диаграмма vs отсортированные столбцы
shares = pd.Series({"Алгебра": 18, "Геометрия": 17, "Анализ": 16, "Статистика": 16, "Логика": 15, "Программирование": 18})

fig, axs = plt.subplots(1, 2, figsize=(11, 4), constrained_layout=True)
axs[0].pie(shares, labels=shares.index, autopct="%d%%")
axs[0].set_title("Pie: что больше — Анализ или Статистика?")
s = shares.sort_values()
axs[1].barh(s.index, s.values, color="tab:blue")
axs[1].set_title("Столбцы: ранжирование видно сразу")
plt.show()

In [ ]:
# (3) Overplotting: 20 000 точек — «чёрное пятно»
pts = np.vstack([rng.normal([0, 0], [1, 1], (10000, 2)),
                 rng.normal([1.5, 1.5], [0.6, 0.6], (10000, 2))])

fig, axs = plt.subplots(1, 3, figsize=(14, 4), constrained_layout=True)
axs[0].scatter(pts[:, 0], pts[:, 1], s=10);                      axs[0].set_title("scatter по умолчанию: пятно")
axs[1].scatter(pts[:, 0], pts[:, 1], s=5, alpha=0.05);           axs[1].set_title("alpha=0.05: видна плотность")
hb = axs[2].hexbin(pts[:, 0], pts[:, 1], gridsize=40, cmap="viridis")
fig.colorbar(hb, ax=axs[2], label="точек в ячейке");             axs[2].set_title("hexbin: считаем точки")
plt.show()

In [ ]:
# (4) Радужная палитра (jet) и перцептивно равномерная (viridis)
g = np.linspace(-2, 3, 200)
X, Y = np.meshgrid(g, g)
Z = np.exp(-(X**2 + Y**2)) + 0.5 * np.exp(-3 * ((X - 1.5) ** 2 + (Y - 1) ** 2))

def to_gray(cmap_name, z):
    rgba = plt.get_cmap(cmap_name)(mpl.colors.Normalize()(z))
    return rgba[..., :3] @ np.array([0.299, 0.587, 0.114])

fig, axs = plt.subplots(2, 2, figsize=(9, 7), constrained_layout=True)
for j, cmap in enumerate(["jet", "viridis"]):
    axs[0, j].imshow(Z, cmap=cmap, origin="lower"); axs[0, j].set_title(cmap)
    axs[1, j].imshow(to_gray(cmap, Z), cmap="gray", origin="lower", vmin=0, vmax=1)
    axs[1, j].set_title(f"{cmap} при чёрно-белой печати")
for ax in axs.ravel(): ax.axis("off")
plt.show()

In [ ]:
# ПЛОХОЙ график — найдите все ошибки
mean_tip = tips.groupby("day", observed=True)["tip"].mean()

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(range(len(mean_tip)), mean_tip.values, color=plt.cm.jet(np.linspace(0, 1, len(mean_tip))))
ax.set_ylim(2.5, 3.3)
ax.grid(True, lw=2)
plt.show()

In [ ]:
# ХОРОШИЙ график: вывод в заголовке, акцент на главном, ось от нуля, ДИ, подписи значений
stat = tips.groupby("day", observed=True)["tip"].agg(["mean", "sem"])
best = stat["mean"].idxmax()
colors = ["tab:blue" if d == best else "lightgray" for d in stat.index]

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(stat.index.astype(str), stat["mean"], yerr=1.96 * stat["sem"], color=colors, capsize=4)
ax.bar_label(bars, labels=[f"{v:.2f}" for v in stat["mean"]], padding=6)
ax.set_ylim(0, stat["mean"].max() * 1.35)
ax.set_title(f"{best} лидирует по чаевым ({stat['mean'].max():.2f} $), но ДИ дней перекрываются", loc="left", fontsize=11)
ax.set_ylabel("Средние чаевые, $")
ax.set_xlabel("")
ax.grid(False)
sns.despine()
fig.text(0.01, -0.02, "Усы — 95% доверительный интервал. Источник: seaborn-data, tips", fontsize=8, color="gray")
plt.show()

In [ ]:
penguins.to_csv("data/penguins.csv", index=False)
print("Сохранено строк:", len(penguins))

### Streamlit